# Validación de 24 cortes en Colab (GPU)

Ejecuta `scripts/run_model.py <modelo> --against naive --folds` sobre la rama indicada y empaqueta los resultados.
Antes de ejecutar: Entorno de ejecución > Cambiar tipo de entorno > GPU (T4). El token de GitHub se pide por teclado y no se guarda en el cuaderno.

In [ ]:
MODEL = "cnn_lstm"  # cnn_lstm | tcn | booster (una por ejecucion)
BRANCH = "main"
REPO = "Cesar2926/Proyecto-IA-Orinoco"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os
from getpass import getpass

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]

token = getpass("Token de GitHub (solo lectura): ")
!git clone --branch {BRANCH} https://{token}@github.com/{REPO}.git orinoco
%cd orinoco
# El token no debe quedar guardado en .git/config
!git remote set-url origin https://github.com/{REPO}.git
del token

In [ ]:
!uv sync
!uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

In [ ]:
# La primera ejecución genera el Parquet del dataset (--write-dataset).
!uv run python scripts/run_model.py {MODEL} --against naive --folds --write-dataset

In [ ]:
import shutil

from google.colab import files

path = shutil.make_archive(f"folds_{MODEL}", "zip", f"data/predictions/{MODEL}/backtest")
files.download(path)